In [9]:
# Importing Libraries

import numpy as np
import pandas as pd
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Joshi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
# Loading Dataset

df = pd.read_csv("cleaned_reviews.csv")

df.head()

,sentiments,cleaned_review,cleaned_review_length,review_score
0,positive,i wish would have gotten one earlier love it a...,19,5.0
1,neutral,i ve learned this lesson again open the packag...,88,1.0
2,neutral,it is so slow and lags find better option,9,2.0
3,neutral,roller ball stopped working within months of m...,12,1.0
4,neutral,i like the color and size but it few days out ...,21,1.0


In [11]:
# Understanding Data

print(df.shape)
print(df['sentiments'].value_counts())
print(df.isnull().sum())

(17340, 4)
sentiments
positive    9503
neutral     6303
negative    1534
Name: count, dtype: int64
sentiments               0
cleaned_review           3
cleaned_review_length    0
review_score             0
dtype: int64


In [12]:
# Renaming Columns for Consistency

df.rename(columns={
    'cleaned_review': 'text',
    'sentiments': 'label'
}, inplace=True)

df.head()

,label,text,cleaned_review_length,review_score
0,positive,i wish would have gotten one earlier love it a...,19,5.0
1,neutral,i ve learned this lesson again open the packag...,88,1.0
2,neutral,it is so slow and lags find better option,9,2.0
3,neutral,roller ball stopped working within months of m...,12,1.0
4,neutral,i like the color and size but it few days out ...,21,1.0


In [13]:
# NLP Preprocessing

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r"\d+", "", text)
    return text

def preprocess_text(text):
    text = clean_text(text)
    
    words = text.split()  # Tokenization
    
    words = [word for word in words if word not in stop_words]
    words = [stemmer.stem(word) for word in words]
    
    return " ".join(words)

df['cleaned_text'] = df['text'].apply(preprocess_text)

df[['text', 'cleaned_text']].head()

,text,cleaned_text
0,i wish would have gotten one earlier love it a...,wish would gotten one earlier love make work l...
1,i ve learned this lesson again open the packag...,learn lesson open packag use product right awa...
2,it is so slow and lags find better option,slow lag find better option
3,roller ball stopped working within months of m...,roller ball stop work within month minim use p...
4,i like the color and size but it few days out ...,like color size day return period hold charg


In [14]:
# Splitting Features and Labels and Train-Test Split

X = df['cleaned_text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Feature Engineering (BoW and TF-IDF)

bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [16]:
# Logistic Regression (TF-IDF)

lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8171856978085352
              precision    recall  f1-score   support

    negative       0.74      0.36      0.49       336
     neutral       0.74      0.81      0.78      1253
    positive       0.88      0.90      0.89      1879

    accuracy                           0.82      3468
   macro avg       0.79      0.69      0.72      3468
weighted avg       0.82      0.82      0.81      3468



In [17]:
# Naive Bayes (BoW)

nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.7165513264129181
              precision    recall  f1-score   support

    negative       0.71      0.27      0.39       336
     neutral       0.65      0.56      0.60      1253
    positive       0.75      0.90      0.82      1879

    accuracy                           0.72      3468
   macro avg       0.70      0.58      0.60      3468
weighted avg       0.71      0.72      0.70      3468



In [18]:
# Model Comparison

results = {
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
}

pd.DataFrame(results)

,Model,Accuracy
0,Logistic Regression,0.817186
1,Naive Bayes,0.716551
2,Decision Tree,0.843426


# Conclusion

- TF-IDF performed better than Bag of Words (BoW) as it considers the importance of words rather than just frequency.

- Decision Tree achieved the highest accuracy in this case, but it may suffer from overfitting on text data.

- Logistic Regression provided stable and balanced performance, making it a reliable choice for sentiment analysis.

- Naive Bayes is fast and efficient but generally less accurate compared to other models.

- Preprocessing steps such as stopword removal, cleaning, and stemming significantly improved model performance.

# Best Pipeline:
TF-IDF + Decision Tree / Logistic Regression